In [1]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline

df = yf.Ticker("AAPL").history(period="3y", interval="1d")
kurse= df["Close"]

df.to_csv("kurse.csv",index=False)

plt.figure(figsize=(12, 5))
plt.plot(kurse.index, kurse, label="Apple-Schlusskurs")
plt.title("Apple – Schlusskurse der letzten drei Jahre")
plt.xlabel("Datum")
plt.ylabel("Kurs in USD")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()





ModuleNotFoundError: No module named 'pandas'

In [1]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

/Users/n1ght/Projekte/chronos-forecasting/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1020 12:28:26.780000 32224 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 170/170 [00:00<00:00, 6736.44it/s]


## Univariate Forecasting

We start with a simple univariate forecasting example using the pandas API.

In [2]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Load data from yfinance Bib
raw_df = yf.Ticker("AAPL").history(period="3y", interval="1d")

#Date aus dem Index in eine Spalte verschieben und umbenennen
context_df = (
    raw_df[["Close"]]
    .reset_index()
    .rename(columns={"Date": "timestamp"})
)

# In datetime umwandeln und Zeitzoneninformation entfernen
# Dabei bleibt das lokale Datum erhalten
context_df["timestamp"] = (
    pd.to_datetime(context_df["timestamp"])
    .dt.tz_localize(None)
)

# Eine gemeinsame ID für die gesamte Zeitreihe
context_df["id"] = "AAPL"

# Ausschließlich Schlusskurse verwenden und zeitlich sortieren
context_df = (
    context_df[["id", "timestamp", "Close"]]
    .dropna(subset=["timestamp", "Close"])
    .sort_values(["id", "timestamp"])
    .reset_index(drop=True)
)

print("Input dataframe shape:", context_df.shape)
print(context_df.head())



ModuleNotFoundError: No module named 'pandas'

In [71]:
pred_df = pipeline.predict_df(context_df, prediction_length=7, quantile_levels=[0.1, 0.5, 0.9],timestamp_column='timestamp', id_column='id', target='Close',freq="B")


print("Output dataframe shape:", pred_df.shape)
display(pred_df.head(7))


Output dataframe shape: (7, 7)


,id,timestamp,target_name,predictions,0.1,0.5,0.9
0,AAPL,2026-10-21,Close,340.892242,334.345001,340.892242,347.241852
1,AAPL,2026-10-22,Close,340.879272,333.426453,340.879272,347.409973
2,AAPL,2026-10-23,Close,340.788116,332.457825,340.788116,347.804077
3,AAPL,2026-10-26,Close,340.595642,332.120148,340.595642,348.536072
4,AAPL,2026-10-27,Close,341.069153,331.310425,341.069153,349.724976
5,AAPL,2026-10-28,Close,340.724274,330.595581,340.724274,350.639069
6,AAPL,2026-10-29,Close,341.378693,330.774109,341.378693,351.582977
